# Methodology v.01 — end-to-end notebook

This notebook **walks through the six-stage pipeline** from *Methodology v.01* using `water_methodology_impl.py`. The code **aligns with the tables and equations where noted below**; see the **Compliance audit** in that module. **Table 4 P5** now includes rolling **min/max** (plus mean/std) at 1h and 6h. For a **single short-sample run** that adds Sec 2.3.2, Table 6 gates, Tier‑1/2 smoke, LUT + DDPG baselines, and Wilcoxon + Bonferroni + Cohen’s *d*, use `water_experiments_small.methodology_first_pass_small(demo_mode=True)`.

1. **Dataset curation & action reconstruction** — **Table 2** RDI roles via `water_rdi_loaders.build_table2_mixed`: **USGS NWIS** IV/DV for DS-1 / DS-4 / DS-5 when online, bundled **WQP-derived DS-2** (Ca+Mg→hardness paired with specific conductance on the same WQP `ActivityIdentifier`), optional `WATER_DS3_CSV`, else synthetic fallbacks; see `m.RDI_TABLE2` / `m.rdi_table2_lines()`. **CUSUM** (Eq. 3) + **inverse titration** labelling (Eq. 4); **Table 3** via `table3_reconstruction_metrics`.  
2. **Physics-grounded environment** — alkalinity / DIC closure `f_titration` (Sec 2.3, Eq. 2 narrative; solver differs from Newton–Raphson in the text); **LSTM surrogate** (Table 5); **Stage 2a** — Sec 2.3.2 + **Table 6** smoke gates before PPO.  
3. **MDP** — 13-D observation layout (Table 8), 11 actions (Table 9), five-term reward (Table 10), γ=0.99, T_max=480 (Sec 4.5).  
4. **PPO** (Table 12) — GAE-λ, clipped objective, entropy schedule hook, curriculum masking (Sec 5.3), MC-Dropout gate (Table 13), running reward norm, hybrid physics→LSTM (Eq. 5), hard pH termination.  
5. **Evaluation** — Tier-1 DCR + **Wilcoxon** vs **RBT, PID, LUT** (Sec 7.2–7.3; Bonferroni α/3 for three comparisons).  
6. **OOD & sim-to-real** — **DS-4** Global Multi-Decadal WQ (GEMS/Water–UN) shift under train scalers (Q2 proxy); **DS-5** path in Stage 2a + Table 4 P4 resampling; full smoke in `methodology_first_pass_small`.

> **Data:** Set `WATER_USE_SYNTH_ONLY=1` to force synthetic Table-2 proxies. Otherwise Stage 1 uses public NWIS + bundled WQP DS-2. Keep chronological splits and train-only scalers (Table 4 P6).  
> **Compute:** `demo_mode=True` shortens RL/LSTM; use `demo_mode=False` for manuscript-scale budgets (Table 12).


In [ ]:
import os, sys
from pathlib import Path

# Ensure the Water/ directory is on the path when running from elsewhere
ROOT = Path.cwd()
if (ROOT / "water_methodology_impl.py").is_file():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "Water" / "water_methodology_impl.py").is_file():
    sys.path.insert(0, str(ROOT.parent / "Water"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import water_methodology_impl as m
print("Loaded water_methodology_impl from:", m.__file__)
print("\nTable 2 — RDI dataset inventory (*Methodology v.01* names):")
for line in m.rdi_table2_lines():
    print(" ", line)


## Stage 1 — Table 2 inventory + CUSUM reconstruction (Sec 2.1–2.4)

Loads **Table 2** frames (`build_table2_mixed`: NWIS + bundled WQP DS-2 when available), then Eq. 3–4 and Table 3 *quality checks* (event rate, non-null fraction).


In [ ]:
import os

DEMO_MODE = True  # False -> manuscript-scale budgets in run_full_pipeline

# Require every Table-2 role to use public data (NWIS + bundled WQP CSVs). Comment out for offline runs.
os.environ["WATER_TABLE2_REQUIRE_REAL"] = "1"

from water_rdi_loaders import build_table2_mixed

frames, TABLE2_FLAGS = build_table2_mixed(demo_mode=DEMO_MODE, synth_module=m)
ds1, ds2, ds3, ds4 = frames["DS-1"], frames["DS-2"], frames["DS-3"], frames["DS-4"]
DS5_IV = frames["DS-5"]
print("Table 2 — public/real where True:", TABLE2_FLAGS)

print(m.rdi_dataset_name("DS-3"), "— compliance positive fraction:", float(ds3["compliance_status"].mean()))
A_T, C_T = m.estimate_AT_CT_from_ds2(ds2, float(ds1["conductivity_uScm"].median()))
acts = m.reconstruct_actions(ds1, A_T, C_T)

print("A_T, C_T (buffering proxies):", A_T, C_T)
print("CUSUM / inverse: non-null action frac =", float((acts != 0).mean()))
ph = ds1["pH"].to_numpy()
print("CUSUM event rate =", float(m.cusum_events(ph).mean()))

t3 = m.table3_reconstruction_metrics(ds1, A_T, C_T)
print("Table 3 (reconstruction diagnostics):", t3)


## Stage 1d — Preprocessing P1–P6 (Table 4)

Chronological 70/15/15; **MinMax** fit on train only; DS-4 kept out of training in `run_full_pipeline`.


In [ ]:
sl_tr, sl_va, sl_te = m.chronological_split(len(ds1))
tr_raw, va_raw, te_raw = ds1.iloc[sl_tr], ds1.iloc[sl_va], ds1.iloc[sl_te]
tr_f, st = m.preprocess_monitor(tr_raw, fit=True)
va_f, _ = m.preprocess_monitor(va_raw, stats=st, fit=False)
tr_f = m.attach_actions(tr_f, acts[sl_tr])
va_f = m.attach_actions(va_f, acts[sl_va])
print(tr_f.head(3))


## Stage 2 — LSTM surrogate (Table 5) + acceptance-style diagnostics (Table 6)

Trains **2×LSTM (256→128)**, dropout **p=0.2**, **L=48**, **Huber δ=1**, **Adam 1e-3** + cosine tail. Validation RMSE is computed in **physical pH** via `pH + ΔpH`.


In [ ]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mm_cols = st["mm_cols"]
tr_ds = m.SeqDS(tr_f, mm_cols, m.L_SEQ)
va_ds = m.SeqDS(va_f, mm_cols, m.L_SEQ)
lstm, dph_scaler, sigma_model = m.train_lstm(
    tr_ds,
    va_ds,
    DEVICE,
    m.L_SEQ,
    epochs=25 if DEMO_MODE else m.LSTM_EPOCHS,
    patience=4 if DEMO_MODE else m.LSTM_PATIENCE,
)
print("LSTM OK on", DEVICE, "| σ_model (ΔpH RMSE) =", sigma_model)


## Stage 2a — Pre-RL gate checks (Sec 2.3.2, Tables 3 & 6)

Methodology **Table 6** expects surrogate acceptance before heavy RL. This cell reports **Sec 2.3.2** one-step simulator error on DS-5 (Hz), **LSTM validation RMSE / MAE(ΔpH)** on the chronological val split, combined **Table 6 pass/fail** (smoke thresholds from `water_experiments_small`), and an **LSTM-on-DS-5** one-step diagnostic (`lstm_ds5_step_metrics`) that routes held-out IoT-style data through the **train-fitted** preprocessor (sim-to-real stress path).


In [ ]:
import water_experiments_small as ex

ds5_hz_gate = m.synth_ds5_hz(100_000 if DEMO_MODE else 150_000)
sec232 = ex.validate_simulator_sec232(
    ds5_hz_gate, A_T, C_T, n_steps=80 if DEMO_MODE else 200
)
lstm_gate = ex.lstm_val_rmse_and_ece(
    lstm, va_ds, dph_scaler, DEVICE, max_batches=24 if DEMO_MODE else 48
)
gates = ex.table6_gates(
    lstm_gate["surrogate_val_RMSE_pH"],
    lstm_gate["surrogate_MAE_dph_residual"],
    sec232["median_abs_dph_err"],
)
print("Sec 2.3.2 (simulator vs DS-5 Hz drift, random one-step actions):", sec232)
print("LSTM val metrics (subset of Table 6):", lstm_gate)
print("Table 6 gates (smoke thresholds):", gates)

ds5_15m_eval = m.ds5_downsample_15m(ds5_hz_gate)
ds5_lstm = ex.lstm_ds5_step_metrics(
    ds5_15m_eval,
    st,
    lstm,
    dph_scaler,
    mm_cols,
    A_T,
    C_T,
    DEVICE,
    max_batches=16 if DEMO_MODE else 40,
)
print("LSTM on DS-5 (15m, train scalers, Table 6 gate-4 proxy):", ds5_lstm)


## Stage 2b — MC-Dropout variance calibration (Table 13)

95th percentile of training predictive variance becomes the **uncertainty gate** threshold during RL.


In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(tr_ds, batch_size=64, shuffle=True)
vars_train = []
for k, (xb, _, _) in enumerate(loader):
    if k > 12:
        break
    vars_train.append(m.mc_predictive_variance(lstm, xb.to(DEVICE), 8 if DEMO_MODE else 20, DEVICE))
unc_p95 = float(np.percentile(vars_train, 95)) if vars_train else None
print("uncertainty p95 =", unc_p95)


## Stage 3–5 — Hybrid MDP + PPO (Eq. 5–6, Tables 10–13)

`WastewaterMDP` switches from **physics** to **LSTM** after `PHYSICS_WARMSTART_STEPS`; applies **curriculum masking** on actions {4,5,9,10}; **MC-Dropout** penalty when variance > p95.


In [ ]:
phys_warm = 6_000 if DEMO_MODE else 100_000
curr_steps = 5_000 if DEMO_MODE else 200_000
total_steps = 12_000 if DEMO_MODE else 5_000_000
rollout = 512 if DEMO_MODE else 2048
minib = 128 if DEMO_MODE else 512

# σ from Table-2 DS-5 (NWIS IV when available); else synthetic Hz trace
if TABLE2_FLAGS.get("DS-5") and len(DS5_IV) >= 200 and "pH" in DS5_IV.columns:
    ds5_hz = DS5_IV
else:
    ds5_hz = m.synth_ds5_hz(3600 if DEMO_MODE else 7200)
sigma_ph, sigma_cond = m.estimate_sigmas_from_ds5(ds5_hz)


def make_env():
    return m.WastewaterMDP(
        lstm,
        A_T,
        C_T,
        DEVICE,
        physics_warm=phys_warm,
        curriculum_steps=curr_steps,
        unc_p95=unc_p95,
        mc_T=8 if DEMO_MODE else 30,
        mm=st["mm"],
        mm_cols=mm_cols,
        dph_scaler=dph_scaler,
        sigma_ph=sigma_ph,
        sigma_model=sigma_model,
        sigma_cond=sigma_cond,
    )

policy = m.ppo_train(
    make_env,
    total_steps=total_steps,
    rollout_len=rollout,
    minibatch=minib,
    device=DEVICE,
    warmup=1500 if DEMO_MODE else 10_000,
    physics_warm=phys_warm,
    curriculum_steps=curr_steps,
    seed=42,
    ent_decay_steps=15_000 if DEMO_MODE else m.ENT_DECAY_STEPS,
)
print("PPO training finished.")


## Stage 5 — Tier-1 evaluation + Wilcoxon (Sec 7.2–7.3, Table 16)

**DCR** on 480-step episodes; pairwise **Wilcoxon** vs rule-based threshold controller; Bonferroni α/4 noted in methodology — here one primary comparison for brevity.


In [ ]:
import water_experiments_small as ex

n_ep = 12 if DEMO_MODE else 80
ppo_dcrs = [m.rollout_policy(policy, make_env(), np.random.default_rng(i), 1, DEVICE)["DCR"] for i in range(n_ep)]


def _episode_dcr(ctrl: str, seed: int) -> float:
    env = make_env()
    rng = np.random.default_rng(seed)
    env.reset(rng)
    phs = [env.ph]
    pidc = m.PIDController() if ctrl == "pid" else None
    for _ in range(m.T_MAX):
        if ctrl == "rbt":
            a = m.rule_based_action(env.ph)
        elif ctrl == "pid":
            a = pidc.action(env.ph)
        elif ctrl == "lut":
            a = ex.lut_greedy_action(env.ph, env.A_T, env.C_T)
        else:
            a = 0
        _, _, done, _ = env.step(a, rng)
        phs.append(env.ph)
        if done:
            break
    ph_arr = np.asarray(phs)
    return float(np.mean((ph_arr >= m.PH_LO) & (ph_arr <= m.PH_HI)) * 100.0)


rbt_dcrs = np.asarray([_episode_dcr("rbt", i) for i in range(n_ep)])
pid_dcrs = np.asarray([_episode_dcr("pid", i + 10_000) for i in range(n_ep)])
lut_dcrs = np.asarray([_episode_dcr("lut", i + 20_000) for i in range(n_ep)])
ppo_dcrs = np.asarray(ppo_dcrs)

alpha_b = 0.05 / 3.0
print("PPO DCR mean:", float(np.mean(ppo_dcrs)), "RBT:", float(np.mean(rbt_dcrs)), "PID:", float(np.mean(pid_dcrs)), "LUT:", float(np.mean(lut_dcrs)))
for name, other in [("RBT", rbt_dcrs), ("PID", pid_dcrs), ("LUT", lut_dcrs)]:
    stat, p = m.wilcoxon_report(ppo_dcrs, other, alpha_b)
    print(f"Wilcoxon PPO vs {name} (Bonferroni α/3): stat={stat}, p={p}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.boxplot([ppo_dcrs, rbt_dcrs, pid_dcrs, lut_dcrs], tick_labels=["PPO", "RBT", "PID", "LUT"])
ax.set_ylabel("DCR (%)")
ax.set_title("Tier-1 discharge compliance rate (proxy, Table 7.2 baselines)")
plt.show()


## Stage 6 — OOD diagnostics (Q2), sim-to-real hook (Q3), full orchestration

**Q2 (Sec 7.1, DS-4):** the **DS-4** frame from Stage 1 (monthly **NWIS DV** when `TABLE2_FLAGS['DS-4']`, else synthetic) is pushed through the **train-only** preprocessor (`fit=False`, `stats=st`) so you can read distribution shift in normalised space vs DS-1 train — the manuscript’s held-out OOD *evaluation* of policies on DS-4 is a larger study; here we surface the data pipeline hook.

**Q3 (DS-5):** Stage 2a already reports simulator + LSTM stress metrics on IoT-style data; the **First pass** cell below adds Tier metrics, DDPG, and Bonferroni tests from `water_experiments_small`.

`run_full_pipeline(demo_mode=...)` (commented cell) chains the same stages for regression testing.


In [ ]:
# Q2 proxy — DS-4 Global Multi-Decadal WQ (GEMS/Water–UN), Table 2, Sec 7.1
print(m.rdi_dataset_name("DS-4"), "| NWIS monthly (real):", TABLE2_FLAGS.get("DS-4"))
# ds4 from Stage 1 (Table 2 mixed loader)
ds4_full = ds4.copy()
ds4_full["DO_mgL"] = 6.0
ds4_full["turbidity_NTU"] = 12.0
ds4_full["temperature_C"] = 20.0
ds4_f, _ = m.preprocess_monitor(ds4_full, stats=st, fit=False)
cols = [c for c in mm_cols if c in tr_f.columns and c in ds4_f.columns]
if cols:
    shift = {c: float(ds4_f[c].mean() - tr_f[c].mean()) for c in cols[:8]}
    print("OOD mean shift (DS-4 − train), first features:", shift)
else:
    print("OOD shift: (no overlapping mm_cols)")


## First pass — full methodology smoke (small sample)

Runs `run_full_pipeline` then Sec 2.3.2, Table 6-style surrogate checks, tier metrics, LUT + DDPG baselines, and four paired tests vs PPO (Bonferroni-corrected). Expect several minutes on CPU.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if (ROOT / "water_experiments_small.py").is_file():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "Water" / "water_experiments_small.py").is_file():
    sys.path.insert(0, str(ROOT.parent / "Water"))

import water_experiments_small as ex

rep = ex.methodology_first_pass_small(demo_mode=True)
print("Sec 2.3.2:", rep["sec232"])
print("LSTM val metrics:", rep["lstm_metrics"])
print("Table 6 gates:", rep["table6_gates"])
print("Tier1 DCR keys:", list(rep["tier1"].keys()))


In [ ]:
# results = m.run_full_pipeline(demo_mode=True)
# print({k: results[k] for k in ("wilcoxon_stat", "wilcoxon_p", "unc_p95", "A_T", "C_T")})


## Optional — full orchestration (longer)

Uncomment to mirror the manuscript-scale configuration (Table 12: **5×10⁶** steps, three seeds **42,123,777** — loop externally).


In [ ]:
# for seed in (42, 123, 777):
#     pol = m.ppo_train(make_env, total_steps=5_000_000, rollout_len=2048, minibatch=512, device=DEVICE, seed=seed)
#     ...
